# Research Paper Companion — Demo Notebook

End-to-end walkthrough of the system: ingestion → retrieval → ask / summarize / compare → eval metrics.

Run from the `research_companion/` project root. See `REPORT.md` for full results and `architecture.md` for the pipeline diagram.

## 1. Setup

Provider config is read from `.env` (Gemini active by default). To switch to local Ollama or FPT Cloud, comment/uncomment the relevant block in `.env`.

In [ ]:
from pathlib import Path
import sys, os, json

# Make the research_companion package importable from the notebook's parent dir
ROOT = Path.cwd() if Path.cwd().name == 'research_companion' else Path.cwd() / 'research_companion'
sys.path.insert(0, str(ROOT.parent))

from research_companion import (
    build_index_from_corpus, RetrievalConfig,
    ask, summarize_section, compare_papers, load_provider,
)
print('package imported OK')

## 2. Build the index (5 papers, ~4,359 chunks)

First run takes ~30 s (sentence-transformer model download + embedding). Re-runs are faster via the HuggingFace cache.

In [ ]:
CORPUS = ROOT / 'corpus'
index = build_index_from_corpus(CORPUS, RetrievalConfig())
for p in index.papers:
    print(f'{p.paper_id:48} | {p.page_count:>3} pp | {p.title[:55]}')

## 3. Load the LLM provider (optional)

If no `OPENAI_API_KEY` is set, the modes fall back to an extractive answer (top-2 chunks formatted as citations).

In [ ]:
provider = load_provider()
print('provider:', 'LLM (' + os.environ.get('MODES_LLM_MODEL', 'default') + ')' if provider else 'extractive fallback')

## 4. Mode 1 — Q&A with citations

`ask()` retrieves the top-k chunks (filtered by paper/section if supplied), runs the LLM with the citation contract, and returns an `AnswerResult`.

In [ ]:
result = ask(index, 'What is Xavier initialization?', provider=provider)
print(result.answer)
print('\n--- citations ---')
for c in result.citations[:3]:
    print(f"[{c['paper_id']} §{c['section']} p.{c['page']}] {c['quote_snippet']}")
print(f'\nretrieval_ms={result.retrieval_latency_ms} gen_ms={result.generation_latency_ms} grounded={result.grounded}')

## 5. Mode 2 — Section summary

In [ ]:
# Pick the math-of-neural-networks paper for a section summary
math_id = next(p.paper_id for p in index.papers if 'math-neural-networks' in p.source_path)
result = summarize_section(index, math_id, 'initialization', provider=provider)
print(result.answer)

## 6. Mode 3 — Compare two papers

In [ ]:
rag_id = next(p.paper_id for p in index.papers if '2005' in p.source_path)
self_rag_id = next(p.paper_id for p in index.papers if 'self-rag' in p.source_path)
result = compare_papers(index, rag_id, self_rag_id,
                        'Compare the retrieval-and-generate design vs the self-critique design.',
                        provider=provider)
print(result.answer)

## 7. Refusal behavior

Two layers: (a) `min_score` retrieval gate refuses OOS queries before the LLM is called; (b) the system prompt instructs the LLM to ask a clarifying question for vague queries.

In [ ]:
oos = ask(index, 'What is the capital of Mongolia?', provider=provider)
print('OOS:', 'refused' if oos.refused else 'answered', '|', oos.refusal_reason)
print('  ->', oos.answer[:120])
vague = ask(index, 'Is bigger better?', provider=provider)
print('\nVAGUE:', 'refused' if vague.refused else 'answered')
print('  ->', vague.answer[:200])

## 8. Eval metrics (read-only — see REPORT.md for analysis)

In [ ]:
import csv
with open(ROOT / 'results' / 'metrics.csv') as f:
    print('SUMMARY')
    for row in csv.DictReader(f):
        for k, v in row.items():
            print(f'  {k:30} {v}')
print('\nBY BUCKET')
with open(ROOT / 'results' / 'metrics_by_bucket.csv') as f:
    for row in csv.DictReader(f):
        print(f"  {row['bucket']:20} n={row['count']:>2} cite={row['citation_coverage_pct']:>6}% refused={row['refused_pct']:>6}% ms={row['mean_latency_ms']}")

## 9. Launch the Gradio app (optional, blocks the notebook)

```python
from research_companion.app import AppState, build_ui
state = AppState()  # ~30–80 s on first run
build_ui(state).launch(server_port=7860)
```

Open http://127.0.0.1:7860 for the four-tab UI.